# Chapter 4: Sensitivity Analysis on Transaction Volume Thresholds

This notebook tests how stable the **XGBoost 2** model is when we change the rules for filtering data. By changing the minimum and maximum transaction limits used to clean the dataset, we test how filtering out low-activity or high-activity wallets affects model accuracy and calibration. Every test and its results are saved to a separate MLflow project (`btc_timezone_analysis_sensitivity_analysis`) so we can easily compare them.

**Key Steps:**
1. **Setup & MLflow Tracking:** Imports required Python packages and connects to the MLflow tracking server.
2. **Evaluation Metrics:** Defines functions to calculate Brier Score, Area Under the Risk-Coverage Curve (AURC), and accuracy at different confidence levels (Top 10%, 25%, and 50%).
3. **Data Setup:** Loads the cleaned dataset, selects the 24-hour activity features (`col_names`), and splits the data into training and test sets.
4. **Cross-Validation Test:** Runs a 5-fold cross-validation test on the XGBoost model, saving the transaction limit tags (`exclusion_num_txs_min`, `exclusion_num_txs_max`), error scores, and the penalized Brier score ($\lambda = 0.5$) to MLflow.

### 1: Environment Initialization & MLflow Tracking Setup

Initializes the required data manipulation, cross-validation, and gradient boosting libraries, and connects to the local MLflow tracking server under the `btc_timezone_analysis_sensitivity_analysis` experiment.

In [1]:
import pandas as pd
import numpy as np
import mlflow

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import accuracy_score, brier_score_loss, auc
from sklearn.preprocessing import label_binarize, LabelEncoder
from sklearn.utils.class_weight import compute_sample_weight

import xgboost as xgb

# Configure Pandas display options to inspect full feature matrices
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 1000)

In [2]:
# Connect to local MLflow server and set the dedicated sensitivity analysis experiment
mlflow.set_tracking_uri("http://127.0.0.1:5000")
mlflow.set_experiment("btc_timezone_analysis_sensitivity_analysis")

<Experiment: artifact_location='/home/matej/btc_timezone_analysis/03_model_training/mlruns_artifacts/3', creation_time=1790779292325, effective_trace_archival_retention=None, experiment_id='3', last_update_time=1790779292325, lifecycle_stage='active', name='btc_timezone_analysis_sensitivity_analysis', tags={}, trace_location=None, workspace='default'>

### 2: Probabilistic & Selective Classification Metrics

Defines the evaluation functions for computing the Area Under the Risk-Coverage Curve (`calculate_aurc`) and aggregating fold-level Brier scores, overall accuracies, and selective abstention accuracies (`get_mlflow_confidence_metrics`) across the 5 cross-validation folds.

In [6]:
def calculate_aurc(y_true, y_prob, y_pred):
    """
    Calculates the Area Under the Risk-Coverage Curve (AURC) to evaluate 
    selective classification performance.
    """
    # 1. Convert to pure NumPy arrays to avoid Pandas index alignment issues
    y_true_np = np.array(y_true)
    y_pred_np = np.array(y_pred)
    
    # 2. Extract maximum predicted class probability as model confidence
    confidences = np.max(y_prob, axis=1)
    
    # 3. Determine which predictions are incorrect (Error = 1, Correct = 0)
    errors = (y_true_np != y_pred_np).astype(int)
    
    # 4. Sort the samples by confidence in descending order
    sorted_indices = np.argsort(-confidences)
    sorted_errors = errors[sorted_indices]
    
    # 5. Calculate Coverage (x-axis: proportion of dataset retained)
    n_samples = len(y_true_np)
    coverages = np.arange(1, n_samples + 1) / n_samples
    
    # 6. Calculate Risk (y-axis: cumulative selective error rate)
    cumulative_errors = np.cumsum(sorted_errors)
    risks = cumulative_errors / np.arange(1, n_samples + 1)
    
    # 7. Add the starting point (0 coverage) to anchor the curve
    coverages = np.insert(coverages, 0, 0.0)
    risks = np.insert(risks, 0, risks[0]) 
    
    # 8. Integrate area under the curve using the trapezoidal rule
    aurc_score = auc(coverages, risks)
    
    return coverages, risks, aurc_score

In [7]:
def get_mlflow_confidence_metrics(
    y_val_true, y_val_pred, y_val_prob, 
    y_train_true, y_train_pred, y_train_prob, 
    target_names
):
    """Computes global and fold-averaged calibration, accuracy, and coverage metrics for MLflow."""
    metrics = {}

    # 1. Global Brier Scores and Overall Accuracy
    metrics['train_brier_score'] = float(brier_score_loss(label_binarize(y_train_true, classes=target_names), y_train_prob))
    metrics['val_brier_score'] = float(brier_score_loss(label_binarize(y_val_true, classes=target_names), y_val_prob))
    metrics['train_accuracy'] = float(accuracy_score(y_train_true, y_train_pred))
    metrics['val_accuracy'] = float(accuracy_score(y_val_true, y_val_pred))
    metrics['samples'] = len(y_val_true)

    fold_metrics_collection = {
        "val_coverage_10_accuracy": [], 
        "val_coverage_25_accuracy": [], 
        "val_coverage_50_accuracy": [], 
        "val_aurc": [],
        "train_aurc": []
    }
    
    n_splits = 5
    
    # Calculate separate fold sizes (concatenated train array is 4x larger than validation)
    val_fold_size = len(y_val_true) // n_splits
    train_fold_size = len(y_train_true) // n_splits
    
    for i in range(n_splits):
        # 1. Validation Slice
        val_start = i * val_fold_size
        val_end = (i + 1) * val_fold_size if i < n_splits - 1 else len(y_val_true)
        
        y_val_true_fold = np.array(y_val_true[val_start:val_end])
        y_val_pred_fold = np.array(y_val_pred[val_start:val_end])
        y_val_prob_fold = np.array(y_val_prob[val_start:val_end])
        
        # 2. Train Slice
        train_start = i * train_fold_size
        train_end = (i + 1) * train_fold_size if i < n_splits - 1 else len(y_train_true)
        
        y_train_true_fold = np.array(y_train_true[train_start:train_end])
        y_train_pred_fold = np.array(y_train_pred[train_start:train_end])
        y_train_prob_fold = np.array(y_train_prob[train_start:train_end])

        # 3. Validation Selective Coverage Metrics (Top 10%, 25%, 50% Confidence)
        confidences = np.max(y_val_prob_fold, axis=1)
        sorted_idx = np.argsort(confidences)[::-1]
        y_true_sorted = y_val_true_fold[sorted_idx]
        y_pred_sorted = y_val_pred_fold[sorted_idx]
        
        coverages = [0.10, 0.25, 0.50]
        for c in coverages:
            k = max(1, int(c * len(y_val_true_fold))) 
            top_true = y_true_sorted[:k]
            top_pred = y_pred_sorted[:k]
            
            c_str = f"val_coverage_{int(c * 100)}"
            fold_metrics_collection[f"{c_str}_accuracy"].append(float(accuracy_score(top_true, top_pred)))
            
        # 4. Validation & Train AURC Integration
        _, _, val_fold_aurc = calculate_aurc(y_val_true_fold, y_val_prob_fold, y_val_pred_fold)
        fold_metrics_collection['val_aurc'].append(float(val_fold_aurc))
        
        _, _, train_fold_aurc = calculate_aurc(y_train_true_fold, y_train_prob_fold, y_train_pred_fold)
        fold_metrics_collection['train_aurc'].append(float(train_fold_aurc))
        
    # Average all selective metrics across the 5 folds
    for key, values in fold_metrics_collection.items():
        metrics[key] = float(np.mean(values))
    
    return metrics

### 3: Data Ingestion & Feature Subset Configuration

Loads the preprocessed dataset, defines the active feature space (`col_names`) for the sensitivity test, and partitions the data into the $75\%$ training/validation split (`random_state = 1019`).

In [9]:
# Load the preprocessed dataset
data = pd.read_csv("/home/matej/btc_timezone_analysis/02_data_preprocessing/database_from_bitcointalk.csv")
print(len(data))
data.columns = data.columns.astype(str)
data.head()

1934


,Unnamed: 0,entity,offset,is_regional_location,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,total,log_total,min_5h_activity,min_5h_activity_sin,min_5h_activity_cos,min_5h_activity_%,lifespan_days,0_weekday_x,0_weekend_x,1_weekday_x,1_weekend_x,2_weekday_x,2_weekend_x,3_weekday_x,3_weekend_x,4_weekday_x,4_weekend_x,5_weekday_x,5_weekend_x,6_weekday_x,6_weekend_x,7_weekday_x,7_weekend_x,8_weekday_x,8_weekend_x,9_weekday_x,9_weekend_x,10_weekday_x,10_weekend_x,11_weekday_x,11_weekend_x,12_weekday_x,12_weekend_x,13_weekday_x,13_weekend_x,14_weekday_x,14_weekend_x,15_weekday_x,15_weekend_x,16_weekday_x,16_weekend_x,17_weekday_x,17_weekend_x,18_weekday_x,18_weekend_x,19_weekday_x,19_weekend_x,20_weekday_x,20_weekend_x,21_weekday_x,21_weekend_x,22_weekday_x,22_weekend_x,23_weekday_x,23_weekend_x,total_weekday_x,total_weekend_x,log_total_weekday_x,log_total_weekend_x,dst_shift_x,macro_region,region,0_weekday_y,0_weekend_y,1_weekday_y,1_weekend_y,2_weekday_y,2_weekend_y,3_weekday_y,3_weekend_y,4_weekday_y,4_weekend_y,5_weekday_y,5_weekend_y,6_weekday_y,6_weekend_y,7_weekday_y,7_weekend_y,8_weekday_y,8_weekend_y,9_weekday_y,9_weekend_y,10_weekday_y,10_weekend_y,11_weekday_y,11_weekend_y,12_weekday_y,12_weekend_y,13_weekday_y,13_weekend_y,14_weekday_y,14_weekend_y,15_weekday_y,15_weekend_y,16_weekday_y,16_weekend_y,17_weekday_y,17_weekend_y,18_weekday_y,18_weekend_y,19_weekday_y,19_weekend_y,20_weekday_y,20_weekend_y,21_weekday_y,21_weekend_y,22_weekday_y,22_weekend_y,23_weekday_y,23_weekend_y,total_weekday_y,total_weekend_y,log_total_weekday_y,log_total_weekend_y,dst_shift_y
0,0,Entity 100,7.0,0,0.020921,0.037657,0.025105,0.050209,0.050209,0.058577,0.033473,0.054393,0.050209,0.046025,0.046025,0.071130,0.037657,0.033473,0.054393,0.054393,0.062762,0.112971,0.037657,0.016736,0.012552,0.016736,0.004184,0.012552,215,5.370638,21,-0.707107,0.707107,0.062762,527.561829,0.022599,0.023256,0.050847,0.011628,0.028249,0.023256,0.050847,0.046512,0.067797,0.011628,0.073446,0.023256,0.039548,0.023256,0.045198,0.069767,0.033898,0.081395,0.056497,0.023256,0.050847,0.034884,0.050847,0.104651,0.028249,0.058140,0.022599,0.058140,0.056497,0.046512,0.073446,0.011628,0.084746,0.011628,0.062147,0.197674,0.033898,0.046512,0.022599,0.011628,0.011299,0.023256,0.016949,0.023256,0.005650,0.011628,0.011299,0.023256,153.0,62.0,5.030438,4.127134,0.0,East Asia & Pacific,East_Asia_Pac,0.022599,0.023256,0.050847,0.011628,0.028249,0.023256,0.050847,0.046512,0.067797,0.011628,0.073446,0.023256,0.039548,0.023256,0.045198,0.069767,0.033898,0.081395,0.056497,0.023256,0.050847,0.034884,0.050847,0.104651,0.028249,0.058140,0.022599,0.058140,0.056497,0.046512,0.073446,0.011628,0.084746,0.011628,0.062147,0.197674,0.033898,0.046512,0.022599,0.011628,0.011299,0.023256,0.016949,0.023256,0.005650,0.011628,0.011299,0.023256,153.0,62.0,5.030438,4.127134,0.0
1,1,Entity 1000,2.0,0,0.107143,0.107143,0.071429,0.035714,0.017857,0.035714,0.053571,0.017857,0.035714,0.035714,0.089286,0.035714,0.053571,0.053571,0.017857,0.017857,0.017857,0.035714,0.017857,0.035714,0.017857,0.035714,0.017857,0.035714,32,3.465736,16,-0.866025,-0.500000,0.107143,125.700324,0.100000,0.066667,0.100000,0.066667,0.080000,0.033333,0.040000,0.033333,0.020000,0.033333,0.020000,0.066667,0.040000,0.066667,0.020000,0.033333,0.040000,0.033333,0.040000,0.033333,0.080000,0.066667,0.040000,0.033333,0.060000,0.033333,0.040000,0.066667,0.020000,0.033333,0.020000,0.033333,0.020000,0.033333,0.040000,0.033333,0.020000,0.033333,0.040000,0.033333,0.020000,0.033333,0.040000,0.033333,0.020000,0.033333,0.040000,0.033333,26.0,6.0,3.258097,1.791759,11.0,Euro-Africa,Euro_Africa,0.100000,0.066667,0.100000,0.066667,0.080000,0.033333,0.040000,0.033333,0.020000,0.033333,0.020000,0.066667,0.040000,0.066667,0.020000,0.033333,0.040000,0.033333,0.040000,0.033333,0.080000,0.066667,0.040000,0.033333,0.060000,0.033333,0.040000,0.066667,0.020000,0.033333,0.020000,0.033333,0.020000,0.033333,0.040000,0.033333,0.020000,0.033333,0.040000,0.0

In [10]:
# Define target variable and select the feature subset under evaluation
X = data
y = data["region"]

col_names = ["0", "1", "2", "3", "4", "5", "6", "7", "8", "9", "10", "11", "12", "13", "14", "15", "16", "17", "18", "19", "20", "21", "22", "23", "dst_shift"]

In [11]:
# Stratified Train-Test Split (Fixed seed = 1019 for consistency across experiments)
num = 1019
print(num)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, stratify=y, random_state=num)

1019


### 4: Cross-Validated Sensitivity Evaluation

Evaluates the regularized `XGBClassifier` across a 5-fold Stratified Cross-Validation split. Logs experimental metadata—including transaction volume exclusion bounds (`exclusion_num_txs_min`, `exclusion_num_txs_max`) and feature set tags—alongside the overfitting-penalized Brier score ($\lambda = 0.5$) to MLflow.

In [ ]:
with mlflow.start_run(run_name="xgboost_model"):
    lambda_par = 0.5  # Overfitting penalty weight
    experiment_name = f"xgboost hyperparameter fitting with lambda={lambda_par} with higher min eta and higher max gamma"
    
    # Log sensitivity analysis metadata and filtration thresholds
    mlflow.set_tag("model_type", experiment_name)
    mlflow.set_tag("exclusion_num_txs_min", 15)
    mlflow.set_tag("exclusion_num_txs_max", "inf")
    mlflow.set_tag("data type", "24 hour transaction distribution")

    kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=num)

    y_val_cv, y_val_cv_pred, y_val_cv_proba = [], [], []
    y_train_cv, y_train_cv_pred, y_train_cv_proba = [], [], []

    le = LabelEncoder()
    y_train_encoded = le.fit_transform(y_train)
    
    pipeline = xgb.XGBClassifier(
        n_estimators=777,
        eta=0.05564936741180356,
        gamma=5.724469359582323,
        reg_lambda=1.0005545286124224,
        max_depth=5,
        min_child_weight=13,
        subsample=0.6649481827307877,
        colsample_bytree=0.9093752666296373,
        random_state=num
    )

    for train_idx, val_idx in kf.split(X_train[col_names], y_train_encoded):
        X_tr, X_va = X_train[col_names].iloc[train_idx], X_train[col_names].iloc[val_idx]
        y_tr, y_va = y_train_encoded[train_idx], y_train_encoded[val_idx]
        
        # Compute fold-specific balanced sample weights and fit model
        sample_weights = compute_sample_weight(class_weight='balanced', y=y_tr)
        pipeline.fit(X_tr, y_tr, sample_weight=sample_weights)

        # Predict and store Training data
        y_train_cv.extend(le.inverse_transform(y_tr))
        y_train_cv_pred.extend(le.inverse_transform(pipeline.predict(X_tr)))
        y_train_cv_proba.extend(pipeline.predict_proba(X_tr))
        
        # Predict and store Validation data
        y_val_cv.extend(le.inverse_transform(y_va))
        y_val_cv_pred.extend(le.inverse_transform(pipeline.predict(X_va)))
        y_val_cv_proba.extend(pipeline.predict_proba(X_va))

    # Convert accumulated predictions to NumPy arrays
    y_train_cv = np.array(y_train_cv)
    y_train_cv_pred = np.array(y_train_cv_pred)
    y_train_cv_proba = np.array(y_train_cv_proba)
    y_val_cv = np.array(y_val_cv)
    y_val_cv_pred = np.array(y_val_cv_pred)
    y_val_cv_proba = np.array(y_val_cv_proba)

    # Compute and log confidence and calibration metrics
    metrics = get_mlflow_confidence_metrics(
        y_val_cv, y_val_cv_pred, y_val_cv_proba, 
        y_train_cv, y_train_cv_pred, y_train_cv_proba, 
        le.inverse_transform(pipeline.classes_)
    )
    mlflow.log_param("lambda", lambda_par)
    mlflow.log_param("random_state train - test split", num)
    mlflow.log_metrics(metrics)
        
    # Compute and log Overfitting-Penalized Brier Score
    final_score = metrics["val_brier_score"] + lambda_par * np.abs(metrics["val_brier_score"] - metrics["train_brier_score"])
    mlflow.log_metric("optuna_penalized_score", final_score)